# Energy Fraud Detection — Colab Runner

**Before running**: set the runtime to **GPU** via `Runtime → Change runtime type → T4 GPU`.

No Google Drive upload needed — the repo is cloned directly from GitHub.

## Step 1 — Clone the repo

In [ ]:
!git clone https://github.com/Vict5/smart-meter-fraud.git

import os, sys
REPO_PATH = '/content/smart-meter-fraud'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

print('Working directory:', os.getcwd())

## Step 2 — Install dependencies

Run once per Colab session (~3 minutes).

In [ ]:
%%capture
!pip install \
    'scikit-learn==1.3.2' \
    'imbalanced-learn==0.11.0' \
    polars pyarrow tqdm \
    pytorch-lightning torchmetrics \
    xgboost shap
print('Installation complete.')

## Step 3 — Verify data files

The pipeline expects processed CSV files in `data/processed/`.

In [ ]:
import pathlib

processed = pathlib.Path('data/processed')
files = sorted(processed.glob('*.csv')) if processed.exists() else []

if not files:
    print('ERROR: data/processed/ is empty or missing.')
else:
    print(f'Found {len(files)} CSV file(s):')
    for f in files:
        print(' ', f.name)

## Step 4 — (Optional) Restore cached models from Drive

Skip this on your first run. On subsequent sessions, run this cell to restore
previously trained models so you skip the ~1 hour autoencoder training.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
#
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree(f'{CACHE}/models',             'models',                  dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/embeddings',         'data/embeddings',         dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/combined_embeddings','data/combined_embeddings', dirs_exist_ok=True)
# print('Models restored from Drive.')

## Step 5 — Enable inline plots

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## Step 6 — Run the pipeline

On the first run the autoencoders train (~600 epochs x 5 datasets, ~1 hr on GPU).
Subsequent runs detect the saved files and skip straight to classification.

| Mode | Description |
|------|-------------|
| `XGBoost` | 4-fold CV XGBoost ensemble (recommended) |
| `RandomForest` | 4-fold CV Random Forest ensemble |
| `DNN_Classifier` | 4-fold CV MLP ensemble |
| `Binary_Classifier` | Threshold anomaly detection (requires `ONLY_REGULAR=True` in config) |

In [ ]:
from src.pipeline import main
main(mode='XGBoost')

In [ ]:
# Other modes:
# main(mode='RandomForest')
# main(mode='DNN_Classifier')
# main(mode='Binary_Classifier')  # only when ONLY_REGULAR=True in config

## Step 7 — (Optional) Save trained models to Drive

Run after the first successful pipeline run to cache models so future sessions
can skip autoencoder training (restore via Step 4).

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
#
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree('models',                  f'{CACHE}/models',              dirs_exist_ok=True)
# shutil.copytree('data/embeddings',         f'{CACHE}/embeddings',          dirs_exist_ok=True)
# shutil.copytree('data/combined_embeddings',f'{CACHE}/combined_embeddings',  dirs_exist_ok=True)
# print('Models saved to Drive.')

## (Optional) Override config hyperparameters

In [ ]:
# Quick smoke-test with fewer epochs — run this BEFORE Step 6
# from src.models.config import Config
# Config.N_EPOCHS_AE = 50
# Config.N_EPOCHS_DNN = 100